## TEXT ONLY ROBERTA MUSTARD

### install

In [ ]:
!pip -q install torchcodec soundfile
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install transformers datasets evaluate accelerate scikit-learn pandas numpy psutil whisper openai-whisper


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 22.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


### import

In [ ]:
import os
import re
import gc
import time
import math
import json
import shutil
import psutil
import random
import pathlib
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)

import whisper
from google.colab import files

### settings

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAMES = [
    "distilroberta-base",
    "bert-base-uncased"
]

WHISPER_MODEL_SIZE = "small"

TEXT_COL = "SENTENCE"
LABEL_COL = "Sarcasm"
KEY_COL = "KEY"
SCENE_COL = "SCENE"

OUTPUT_ROOT = "/content/mustard_models"
VIDEO_ROOT = "/content/mustard_videos"

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(VIDEO_ROOT, exist_ok=True)

print("DEVICE:", DEVICE)

DEVICE: cuda


### download mustard++_text.csv

In [ ]:
uploaded = files.upload()
csv_name = next(iter(uploaded.keys()))
df_raw = pd.read_csv(csv_name)
print(df_raw.shape)
df_raw.head()

Saving mustard++_text.csv to mustard++_text (2).csv
(6041, 12)


,SCENE,KEY,SENTENCE,END_TIME,SPEAKER,SHOW,Sarcasm,Sarcasm_Type,Implicit_Emotion,Explicit_Emotion,Valence,Arousal
0,1_10004,1_10004_c_00,"Well, I'm sure that, uh, you...\nhave a lot of...",0:06,PERSON,BBT,NaN,NaN,NaN,NaN,NaN,NaN
1,1_10004,1_10004_c_01,Who was he?,0:08,SHELDON,BBT,NaN,NaN,NaN,NaN,NaN,NaN
2,1_10004,1_10004_c_02,His name is Ron.\nI met him at my prayer group.,0:12,PERSON,BBT,NaN,NaN,NaN,NaN,NaN,NaN
3,1_10004,1_10004_c_03,How long have you been involved with him?,0:14,SHELDON,BBT,NaN,NaN,NaN,NaN,NaN,NaN
4,1_10004,1_10004_c_04,A few months.,0:16,PERSON,BBT,NaN,NaN,NaN,NaN,NaN,NaN


### clean dataset

In [ ]:
def normalize_label(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    mapping = {
        "1": 1,
        "true": 1,
        "yes": 1,
        "sarcastic": 1,
        "sarcasm": 1,

        "0": 0,
        "false": 0,
        "no": 0,
        "non-sarcastic": 0,
        "nonsarcastic": 0,
        "not sarcasm": 0
    }

    if s in mapping:
        return mapping[s]

    try:
        num = int(float(s))
        if num in [0, 1]:
            return num
    except:
        pass

    return np.nan


def prepare_mustard_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in [TEXT_COL, LABEL_COL, KEY_COL, SCENE_COL]:
        if col not in df.columns:
            raise ValueError(f"В CSV нет колонки: {col}")

    df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
    df["label"] = df[LABEL_COL].apply(normalize_label)

    df = df[df[TEXT_COL].notna()]
    df = df[df[TEXT_COL] != ""]
    df = df[df["label"].notna()]

    df["label"] = df["label"].astype(int)
    df = df[[SCENE_COL, KEY_COL, TEXT_COL, "label"]].drop_duplicates().reset_index(drop=True)

    return df


df = prepare_mustard_dataframe(df_raw)

print("Размер после очистки:", df.shape)
print(df["label"].value_counts(dropna=False))
df.head()

Размер после очистки: (1202, 4)
label
0    601
1    601
Name: count, dtype: int64


,SCENE,KEY,SENTENCE,label
0,1_10004,1_10004_u,"And of those few months, how long have you bee...",0
1,1_10009,1_10009_u,"Let the dead man talk. So, why do you think that?",0
2,1_1001,1_1001_u,"What else? Sell it on eBay as ""slightly used.""",0
3,1_1003,1_1003_u,"Good idea, sit with her. Hold her, comfort her...",1
4,1_10190,1_10190_u,"Well, now that I've given up string theory, I'...",0


 ### Train / validation / test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (841, 4)
val: (180, 4)
test: (181, 4)


### Hugging Face Dataset

In [ ]:
dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['SCENE', 'KEY', 'SENTENCE', 'label'],
        num_rows: 841
    })
    validation: Dataset({
        features: ['SCENE', 'KEY', 'SENTENCE', 'label'],
        num_rows: 180
    })
    test: Dataset({
        features: ['SCENE', 'KEY', 'SENTENCE', 'label'],
        num_rows: 181
    })
})

### Quality metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

### Tokenization

In [ ]:
def make_tokenize_fn(tokenizer):
    def tokenize_batch(batch):
        return tokenizer(
            batch[TEXT_COL],
            truncation=True,
            max_length=MAX_LENGTH
        )
    return tokenize_batch

### RAM / GPU / FLOPs / latency functions

In [ ]:
def get_ram_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)


def get_gpu_peak_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


def reset_gpu_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def estimate_model_flops(model, tokenizer, seq_len=128, batch_size=1):
    try:
        text = "hello " * max(1, seq_len // 2)
        inputs = tokenizer(
            [text] * batch_size,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=seq_len
        )
        if hasattr(model, "floating_point_ops"):
            flops = model.floating_point_ops(inputs)
            return int(flops)
        return None
    except Exception:
        return None


def measure_classifier_latency(model, tokenizer, texts, batch_size=16, max_length=128):
    model.eval()
    latencies = []
    preds_all = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(DEVICE)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(**inputs)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()

        batch_latency = (end - start) / len(batch_texts)
        latencies.extend([batch_latency] * len(batch_texts))

        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy().tolist()
        preds_all.extend(preds)

    return preds_all, latencies

### DistilRoberta base

In [ ]:
MODEL_NAME = "distilroberta-base"
OUTPUT_DIR = "/content/mustard_distilroberta"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Model:", MODEL_NAME)
print("Output dir:", OUTPUT_DIR)

Model: distilroberta-base
Output dir: /content/mustard_distilroberta


### latency, memory, FLOPs

In [ ]:
def get_ram_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)


def reset_gpu_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def get_gpu_peak_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return None


def estimate_model_flops(model, tokenizer, seq_len=128, batch_size=1):
    try:
        sample_text = "hello " * max(1, seq_len // 2)
        inputs = tokenizer(
            [sample_text] * batch_size,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=seq_len
        )
        if hasattr(model, "floating_point_ops"):
            flops = model.floating_point_ops(inputs)
            return int(flops)
        return None
    except Exception:
        return None


def measure_classifier_latency(model, tokenizer, texts, batch_size=16, max_length=128):
    model.eval()
    all_preds = []
    all_latencies = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(DEVICE)

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        start = time.perf_counter()
        with torch.no_grad():
            outputs = model(**inputs)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()

        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy().tolist()
        batch_latency = (end - start) / len(batch_texts)

        all_preds.extend(preds)
        all_latencies.extend([batch_latency] * len(batch_texts))

    return all_preds, all_latencies

### Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized = dataset_dict.map(tokenize_batch, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "label"]
tokenized.set_format(type="torch", columns=cols_to_keep)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/841 [00:00<?, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['SCENE', 'KEY', 'SENTENCE', 'label', 'input_ids', 'attention_mask'],
        num_rows: 841
    })
    validation: Dataset({
        features: ['SCENE', 'KEY', 'SENTENCE', 'label', 'input_ids', 'attention_mask'],
        num_rows: 180
    })
    test: Dataset({
        features: ['SCENE', 'KEY', 'SENTENCE', 'label', 'input_ids', 'attention_mask'],
        num_rows: 181
    })
})

### create model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(DEVICE)

print(model.__class__.__name__)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification


### params

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    save_total_limit=2
)

### Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

### train and save model

In [ ]:
reset_gpu_peak()
train_ram_before = get_ram_mb()
train_start = time.perf_counter()

trainer.train()

train_end = time.perf_counter()
train_ram_after = get_ram_mb()
train_gpu_peak = get_gpu_peak_mb()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

train_stats = {
    "model_name": MODEL_NAME,
    "train_time_sec": train_end - train_start,
    "train_ram_before_mb": train_ram_before,
    "train_ram_after_mb": train_ram_after,
    "train_gpu_peak_mb": train_gpu_peak
}

print(json.dumps(train_stats, ensure_ascii=False, indent=2))

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.489583,0.755696,0.600000,0.636364,0.466667,0.538462
2,0.304238,1.035416,0.572222,0.618182,0.377778,0.468966
3,0.182287,1.193534,0.577778,0.606061,0.444444,0.512821
4,0.104926,1.443238,0.611111,0.631579,0.533333,0.578313
5,0.067505,1.566622,0.555556,0.575758,0.422222,0.487179


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model_name": "distilroberta-base",
  "train_time_sec": 89.74050975299997,
  "train_ram_before_mb": 1467.5234375,
  "train_ram_after_mb": 1616.87109375,
  "train_gpu_peak_mb": 1327.6884765625
}


### preliminary text test

In [ ]:
eval_metrics = trainer.evaluate(tokenized["test"])
print(json.dumps(eval_metrics, ensure_ascii=False, indent=2))

{
  "eval_loss": 1.2088426351547241,
  "eval_accuracy": 0.6519337016574586,
  "eval_precision": 0.651685393258427,
  "eval_recall": 0.6444444444444445,
  "eval_f1": 0.6480446927374302,
  "eval_runtime": 0.3027,
  "eval_samples_per_second": 597.864,
  "eval_steps_per_second": 39.637,
  "epoch": 5.0
}


### latency test

In [ ]:
def estimate_distilroberta_flops_per_sample(seq_len=128):
    num_layers = 6
    hidden_size = 768
    intermediate_size = 3072
    num_heads = 12

    d = hidden_size
    s = seq_len
    f = intermediate_size

    # Q,K,V + output projection
    attention_linear = 4 * s * d * d

    # attention matrix ops
    attention_scores = 2 * s * s * d

    # FFN
    ffn = 2 * s * d * f

    flops_per_layer = attention_linear + attention_scores + ffn
    total_flops = num_layers * flops_per_layer

    return int(total_flops)

test_texts = test_df[TEXT_COL].tolist()
test_labels = test_df["label"].tolist()

reset_gpu_peak()
infer_ram_before = get_ram_mb()
infer_start = time.perf_counter()

test_preds, latencies = measure_classifier_latency(
    model=model,
    tokenizer=tokenizer,
    texts=test_texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

infer_end = time.perf_counter()
infer_ram_after = get_ram_mb()
infer_gpu_peak = get_gpu_peak_mb()

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels,
    test_preds,
    average="binary",
    zero_division=0
)
acc = accuracy_score(test_labels, test_preds)

flops = estimate_distilroberta_flops_per_sample(seq_len=MAX_LENGTH)

inference_stats = {
    "test_accuracy": acc,
    "test_precision": precision,
    "test_recall": recall,
    "test_f1": f1,
    "classifier_infer_total_sec": infer_end - infer_start,
    "classifier_latency_mean_ms": float(np.mean(latencies) * 1000),
    "classifier_latency_std_ms": float(np.std(latencies) * 1000),
    "classifier_ram_before_mb": infer_ram_before,
    "classifier_ram_after_mb": infer_ram_after,
    "classifier_gpu_peak_mb": infer_gpu_peak,
    "approx_flops_per_sample": flops
}

print(json.dumps(inference_stats, ensure_ascii=False, indent=2))

{
  "test_accuracy": 0.6519337016574586,
  "test_precision": 0.651685393258427,
  "test_recall": 0.6444444444444445,
  "test_f1": 0.6480446927374302,
  "classifier_infer_total_sec": 0.31031541699849186,
  "classifier_latency_mean_ms": 1.5243823093728603,
  "classifier_latency_std_ms": 0.4576907816856413,
  "classifier_ram_before_mb": 2602.28125,
  "classifier_ram_after_mb": 2602.28125,
  "classifier_gpu_peak_mb": 1919.8076171875,
  "approx_flops_per_sample": 5586812928
}


### save metrics

In [ ]:
all_stats = {}
all_stats.update(train_stats)
all_stats.update(eval_metrics)
all_stats.update(inference_stats)

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(all_stats, f, ensure_ascii=False, indent=2)

pd.DataFrame([all_stats]).to_csv(
    os.path.join(OUTPUT_DIR, "metrics.csv"),
    index=False
)

print("Saved metrics to:", OUTPUT_DIR)

Saved metrics to: /content/mustard_distilroberta


### save predictions

In [ ]:
test_preds_df = test_df.copy()
test_preds_df["pred"] = test_preds
test_preds_df["correct"] = (test_preds_df["label"] == test_preds_df["pred"]).astype(int)

test_preds_df.to_csv(
    os.path.join(OUTPUT_DIR, "test_predictions.csv"),
    index=False
)

test_preds_df.head()

,SCENE,KEY,SENTENCE,label,pred,correct
187,1_5786,1_5786_u,"All right, so technically it's not a dinner da...",0,0,1
942,2_416,2_416_u,Were you so late because you were burying this...,1,0,0
1065,2_570,2_570_u,"Oh my god, where are all the men?",0,0,1
62,1_12275,1_12275_u,"Uh, no. I overheard Bernadette tell Howard she...",0,0,1
806,2_225,2_225_u,"I don't know, something girlie.",0,1,0


### save embeddings for hybrid architecture experiment

In [ ]:
def extract_embeddings(texts, tokenizer, base_model, batch_size=16, max_length=128):
    base_model.eval()
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(DEVICE)

        with torch.no_grad():
            outputs = base_model(
                **inputs,
                output_hidden_states=True,
                return_dict=True
            )

        last_hidden = outputs.hidden_states[-1]
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_hidden = last_hidden * attention_mask
        summed = masked_hidden.sum(dim=1)
        counts = attention_mask.sum(dim=1).clamp(min=1)
        mean_pooled = summed / counts

        all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

### train/val/test embeddings

In [ ]:
base_model = model.roberta

train_texts = train_df[TEXT_COL].tolist()
val_texts = val_df[TEXT_COL].tolist()
test_texts = test_df[TEXT_COL].tolist()

train_embeddings = extract_embeddings(
    train_texts,
    tokenizer,
    base_model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

val_embeddings = extract_embeddings(
    val_texts,
    tokenizer,
    base_model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

test_embeddings = extract_embeddings(
    test_texts,
    tokenizer,
    base_model,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

print("train:", train_embeddings.shape)
print("val:", val_embeddings.shape)
print("test:", test_embeddings.shape)

train: (841, 768)
val: (180, 768)
test: (181, 768)


### save all embeddings and split

In [ ]:
np.save(os.path.join(OUTPUT_DIR, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(OUTPUT_DIR, "val_embeddings.npy"), val_embeddings)
np.save(os.path.join(OUTPUT_DIR, "test_embeddings.npy"), test_embeddings)

train_df.to_csv(os.path.join(OUTPUT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_split.csv"), index=False)

print("Embeddings and splits saved to:", OUTPUT_DIR)

Embeddings and splits saved to: /content/mustard_distilroberta


### archive all

In [ ]:
archive_path = shutil.make_archive(
    base_name="/content/mustard_distilroberta_results",
    format="zip",
    root_dir=OUTPUT_DIR
)

print("Archive created:", archive_path)

Archive created: /content/mustard_distilroberta_results.zip


### download archive

In [ ]:
files.download("/content/mustard_distilroberta_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## TEST EVALUATION USING WHISPER

### get Whisper

In [ ]:
!pip -q install openai-whisper

### import

In [ ]:
import whisper
from google.colab import drive

### google drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


### videos

In [ ]:
!pip install gdown

In [ ]:
!pip -q install PyDrive2

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

print("Drive auth OK")

Drive auth OK


In [ ]:
FOLDER_ID = "1xcvGIPGzrcNKkNBwBSoik9sh_SDSH2ov"
DOWNLOAD_DIR = "/content/test_videos"

import os
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Folder ID:", FOLDER_ID)
print("Download dir:", DOWNLOAD_DIR)

Folder ID: 1xcvGIPGzrcNKkNBwBSoik9sh_SDSH2ov
Download dir: /content/test_videos


In [ ]:
test_keys = test_df[KEY_COL].astype(str).unique().tolist()
needed_filenames = {f"{k}.mp4" for k in test_keys}

print("Unique test keys:", len(test_keys))
print("Need to download:", len(needed_filenames))
print(list(sorted(needed_filenames))[:10])

Unique test keys: 181
Need to download: 181
['1_105_u.mp4', '1_10748_u.mp4', '1_10890_u.mp4', '1_11006_u.mp4', '1_11046_u.mp4', '1_11098_u.mp4', '1_11526_u.mp4', '1_11699_u.mp4', '1_11889_u.mp4', '1_1189_u.mp4']


In [ ]:
query = f"'{FOLDER_ID}' in parents and trashed=false"

file_list = drive.ListFile({
    "q": query,
    "supportsAllDrives": True,
    "includeItemsFromAllDrives": True
}).GetList()

print("Files visible in folder:", len(file_list))

matched_files = [f for f in file_list if f["title"] in needed_filenames]

print("Matched test videos:", len(matched_files))
print([f["title"] for f in matched_files[:10]])

Files visible in folder: 1203
Matched test videos: 181
['1_60_u.mp4', '3_S06E02_171_u.mp4', '2_96_u.mp4', '2_72_u.mp4', '2_426_u.mp4', '1_S12E09_081_u.mp4', '1_S10E03_311_u.mp4', '2_454_u.mp4', '1_S12E04_195_u.mp4', '2_400_u.mp4']


In [ ]:
downloaded = []
failed = []

for i, f in enumerate(matched_files, 1):
    title = f["title"]
    out_path = os.path.join(DOWNLOAD_DIR, title)

    try:
        if not os.path.exists(out_path):
            f.GetContentFile(out_path)
        downloaded.append(title)

        if i % 20 == 0 or i == len(matched_files):
            print(f"Downloaded {i}/{len(matched_files)}")
    except Exception as e:
        failed.append((title, str(e)))

print("Done")
print("Downloaded:", len(downloaded))
print("Failed:", len(failed))

Downloaded 20/181
Downloaded 40/181
Downloaded 60/181
Downloaded 80/181
Downloaded 100/181
Downloaded 120/181
Downloaded 140/181
Downloaded 160/181
Downloaded 180/181
Downloaded 181/181
Done
Downloaded: 181
Failed: 0


In [ ]:
print(os.listdir(DOWNLOAD_DIR)[:10])

['1_12331_u.mp4', '1_7675_u.mp4', '1_S11E24_338_u.mp4', '1_S09E21_024_u.mp4', '1_S11E06_410_u.mp4', '2_372_u.mp4', '2_464_u.mp4', '2_272_u.mp4', '1_S12E07_169_u.mp4', '1_S11E19_143_u.mp4']


In [ ]:
import pathlib

VIDEO_EXTS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".flv"}

def index_video_files(video_root: str):
    video_index = {}
    for root, _, files in os.walk(video_root):
        for fname in files:
            ext = pathlib.Path(fname).suffix.lower()
            if ext in VIDEO_EXTS:
                full_path = os.path.join(root, fname)
                stem = pathlib.Path(fname).stem
                video_index[stem] = full_path
    return video_index

video_index = index_video_files(DOWNLOAD_DIR)
print("Indexed videos:", len(video_index))
print(list(video_index.items())[:5])

Indexed videos: 181
[('1_12331_u', '/content/test_videos/1_12331_u.mp4'), ('1_7675_u', '/content/test_videos/1_7675_u.mp4'), ('1_S11E24_338_u', '/content/test_videos/1_S11E24_338_u.mp4'), ('1_S09E21_024_u', '/content/test_videos/1_S09E21_024_u.mp4'), ('1_S11E06_410_u', '/content/test_videos/1_S11E06_410_u.mp4')]


In [ ]:
test_video_df = test_df.copy()
test_video_df["video_id"] = test_video_df[KEY_COL].astype(str)
test_video_df["video_path"] = test_video_df["video_id"].map(video_index)

print("All test rows:", len(test_video_df))
print("Matched local videos:", test_video_df["video_path"].notna().sum())

test_video_df = test_video_df[test_video_df["video_path"].notna()].reset_index(drop=True)
test_video_df.head()

All test rows: 181
Matched local videos: 181


,SCENE,KEY,SENTENCE,label,video_id,video_path
0,1_5786,1_5786_u,"All right, so technically it's not a dinner da...",0,1_5786_u,/content/test_videos/1_5786_u.mp4
1,2_416,2_416_u,Were you so late because you were burying this...,1,2_416_u,/content/test_videos/2_416_u.mp4
2,2_570,2_570_u,"Oh my god, where are all the men?",0,2_570_u,/content/test_videos/2_570_u.mp4
3,1_12275,1_12275_u,"Uh, no. I overheard Bernadette tell Howard she...",0,1_12275_u,/content/test_videos/1_12275_u.mp4
4,2_225,2_225_u,"I don't know, something girlie.",0,2_225_u,/content/test_videos/2_225_u.mp4


###  download whisper

In [ ]:
!pip -q install openai-whisper

### imports for ASR and downloads

In [ ]:
import whisper
import json
import numpy as np
import pandas as pd
import time
import os

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

### whisper small


In [ ]:
whisper_model = whisper.load_model("small", device=DEVICE)
print("Whisper loaded on", DEVICE)

100%|███████████████████████████████████████| 461M/461M [00:09<00:00, 52.6MiB/s]


Whisper loaded on cuda


### video transcribe func

In [ ]:
def transcribe_video_whisper(video_path, whisper_model):
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    result = whisper_model.transcribe(
        video_path,
        language="en",
        fp16=torch.cuda.is_available()
    )

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    text = result["text"].strip()
    elapsed = end - start

    return text, elapsed

### distilroberta classification function

In [ ]:
def predict_text_roberta(text, model, tokenizer, max_length=128):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    ).to(DEVICE)

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        outputs = model(**inputs)

    if DEVICE == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    pred = torch.argmax(outputs.logits, dim=-1).item()
    elapsed = end - start

    return pred, elapsed

### flops here for classifier

In [ ]:
classifier_flops = estimate_model_flops(
    model=model,
    tokenizer=tokenizer,
    seq_len=MAX_LENGTH,
    batch_size=1
)

print("Approx classifier FLOPs per sample:", classifier_flops)

Approx classifier FLOPs per sample: None


### video -> whisper -> distilroberta

In [ ]:
reset_gpu_peak()
pipeline_ram_before = get_ram_mb()
pipeline_start = time.perf_counter()

y_true = []
y_pred = []

whisper_times = []
roberta_times = []
end_to_end_times = []

transcripts = []
source_video_paths = []
source_keys = []

for _, row in test_video_df.iterrows():
    true_label = int(row["label"])
    video_path = row["video_path"]
    key_value = row[KEY_COL]

    t0 = time.perf_counter()

    transcript, whisper_time = transcribe_video_whisper(video_path, whisper_model)
    pred, roberta_time = predict_text_roberta(
        transcript,
        model,
        tokenizer,
        max_length=MAX_LENGTH
    )

    t1 = time.perf_counter()

    y_true.append(true_label)
    y_pred.append(pred)

    whisper_times.append(whisper_time)
    roberta_times.append(roberta_time)
    end_to_end_times.append(t1 - t0)

    transcripts.append(transcript)
    source_video_paths.append(video_path)
    source_keys.append(key_value)

pipeline_end = time.perf_counter()
pipeline_ram_after = get_ram_mb()
pipeline_gpu_peak = get_gpu_peak_mb()

print("Processed samples:", len(y_true))

Processed samples: 181


### metrics pipeline

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="binary",
    zero_division=0
)

acc = accuracy_score(y_true, y_pred)

print("Pipeline quality metrics:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")

Pipeline quality metrics:
Accuracy : 0.6022
Precision: 0.6154
Recall   : 0.5333
F1       : 0.5714


### time, memory, latency, FLOPs

In [ ]:
classifier_flops = estimate_distilroberta_flops_per_sample(seq_len=MAX_LENGTH)

pipeline_stats = {
    "n_samples_with_video": len(y_true),

    "pipeline_accuracy": acc,
    "pipeline_precision": precision,
    "pipeline_recall": recall,
    "pipeline_f1": f1,

    "pipeline_total_time_sec": pipeline_end - pipeline_start,

    "whisper_total_time_sec": float(np.sum(whisper_times)),
    "whisper_mean_latency_ms": float(np.mean(whisper_times) * 1000),
    "whisper_std_latency_ms": float(np.std(whisper_times) * 1000),

    "roberta_total_time_sec": float(np.sum(roberta_times)),
    "roberta_mean_latency_ms": float(np.mean(roberta_times) * 1000),
    "roberta_std_latency_ms": float(np.std(roberta_times) * 1000),

    "end_to_end_total_time_sec": float(np.sum(end_to_end_times)),
    "end_to_end_mean_latency_ms": float(np.mean(end_to_end_times) * 1000),
    "end_to_end_std_latency_ms": float(np.std(end_to_end_times) * 1000),

    "pipeline_ram_before_mb": pipeline_ram_before,
    "pipeline_ram_after_mb": pipeline_ram_after,
    "pipeline_gpu_peak_mb": pipeline_gpu_peak,

    "classifier_approx_flops_per_sample": classifier_flops
}

### classification report

In [ ]:
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

              precision    recall  f1-score   support

           0     0.5922    0.6703    0.6289        91
           1     0.6154    0.5333    0.5714        90

    accuracy                         0.6022       181
   macro avg     0.6038    0.6018    0.6001       181
weighted avg     0.6037    0.6022    0.6003       181



### download transcript, prediction and path

In [ ]:
whisper_test_results_df = test_video_df.copy()
whisper_test_results_df["whisper_text"] = transcripts
whisper_test_results_df["pred"] = y_pred
whisper_test_results_df["correct"] = (whisper_test_results_df["label"] == whisper_test_results_df["pred"]).astype(int)

whisper_test_results_path = os.path.join(OUTPUT_DIR, "whisper_test_predictions.csv")
whisper_test_results_df.to_csv(whisper_test_results_path, index=False)

print("Saved:", whisper_test_results_path)
whisper_test_results_df.head()

Saved: /content/mustard_distilroberta/whisper_test_predictions.csv


,SCENE,KEY,SENTENCE,label,video_id,video_path,whisper_text,pred,correct
0,1_5786,1_5786_u,"All right, so technically it's not a dinner da...",0,1_5786_u,/content/test_videos/1_5786_u.mp4,"All right, so technically, it's not a dinner d...",0,1
1,2_416,2_416_u,Were you so late because you were burying this...,1,2_416_u,/content/test_videos/2_416_u.mp4,Were you so late because you were burying this...,0,0
2,2_570,2_570_u,"Oh my god, where are all the men?",0,2_570_u,/content/test_videos/2_570_u.mp4,"Oh my God, where are all the men?",0,1
3,1_12275,1_12275_u,"Uh, no. I overheard Bernadette tell Howard she...",0,1_12275_u,/content/test_videos/1_12275_u.mp4,"But no, I overheard Bernadette tell Howard she...",0,1
4,2_225,2_225_u,"I don't know, something girlie.",0,2_225_u,/content/test_videos/2_225_u.mp4,"I don't know, something girly.",0,1


### json and csv

In [ ]:
whisper_test_results_df = test_video_df.copy()
whisper_test_results_df["whisper_text"] = transcripts
whisper_test_results_df["pred"] = y_pred
whisper_test_results_df["correct"] = (whisper_test_results_df["label"] == whisper_test_results_df["pred"]).astype(int)

whisper_test_results_path = os.path.join(OUTPUT_DIR, "whisper_test_predictions.csv")
whisper_test_results_df.to_csv(whisper_test_results_path, index=False)

print("Saved:", whisper_test_results_path)
whisper_test_results_df.head()

Saved: /content/mustard_distilroberta/whisper_test_predictions.csv


,SCENE,KEY,SENTENCE,label,video_id,video_path,whisper_text,pred,correct
0,1_5786,1_5786_u,"All right, so technically it's not a dinner da...",0,1_5786_u,/content/test_videos/1_5786_u.mp4,"All right, so technically, it's not a dinner d...",0,1
1,2_416,2_416_u,Were you so late because you were burying this...,1,2_416_u,/content/test_videos/2_416_u.mp4,Were you so late because you were burying this...,0,0
2,2_570,2_570_u,"Oh my god, where are all the men?",0,2_570_u,/content/test_videos/2_570_u.mp4,"Oh my God, where are all the men?",0,1
3,1_12275,1_12275_u,"Uh, no. I overheard Bernadette tell Howard she...",0,1_12275_u,/content/test_videos/1_12275_u.mp4,"But no, I overheard Bernadette tell Howard she...",0,1
4,2_225,2_225_u,"I don't know, something girlie.",0,2_225_u,/content/test_videos/2_225_u.mp4,"I don't know, something girly.",0,1


### only whisper transcripts

In [ ]:
pipeline_stats_json_path = os.path.join(OUTPUT_DIR, "whisper_pipeline_metrics.json")
pipeline_stats_csv_path = os.path.join(OUTPUT_DIR, "whisper_pipeline_metrics.csv")

with open(pipeline_stats_json_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_stats, f, ensure_ascii=False, indent=2)

pd.DataFrame([pipeline_stats]).to_csv(pipeline_stats_csv_path, index=False)

print("Saved:", pipeline_stats_json_path)
print("Saved:", pipeline_stats_csv_path)

Saved: /content/mustard_distilroberta/whisper_pipeline_metrics.json
Saved: /content/mustard_distilroberta/whisper_pipeline_metrics.csv


### original text vs whisper text

In [ ]:
compare_df = whisper_test_results_df[[KEY_COL, TEXT_COL, "whisper_text", "label", "pred"]].copy()
compare_path = os.path.join(OUTPUT_DIR, "original_vs_whisper_text.csv")
compare_df.to_csv(compare_path, index=False)

print("Saved:", compare_path)
compare_df.head()

Saved: /content/mustard_distilroberta/original_vs_whisper_text.csv


,KEY,SENTENCE,whisper_text,label,pred
0,1_5786_u,"All right, so technically it's not a dinner da...","All right, so technically, it's not a dinner d...",0,0
1,2_416_u,Were you so late because you were burying this...,Were you so late because you were burying this...,1,0
2,2_570_u,"Oh my god, where are all the men?","Oh my God, where are all the men?",0,0
3,1_12275_u,"Uh, no. I overheard Bernadette tell Howard she...","But no, I overheard Bernadette tell Howard she...",0,0
4,2_225_u,"I don't know, something girlie.","I don't know, something girly.",0,0


### text-only + whisper-pipeline

In [ ]:
final_summary = {}

if "all_stats" in globals():
    for k, v in all_stats.items():
        final_summary[f"text_only_{k}"] = v

for k, v in pipeline_stats.items():
    final_summary[f"whisper_pipeline_{k}"] = v

final_summary_path_json = os.path.join(OUTPUT_DIR, "final_summary.json")
final_summary_path_csv = os.path.join(OUTPUT_DIR, "final_summary.csv")

with open(final_summary_path_json, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

pd.DataFrame([final_summary]).to_csv(final_summary_path_csv, index=False)

print("Saved:", final_summary_path_json)
print("Saved:", final_summary_path_csv)

Saved: /content/mustard_distilroberta/final_summary.json
Saved: /content/mustard_distilroberta/final_summary.csv


all text only stats

In [ ]:
summary_rows = []

# -------- TEXT ONLY (gold transcripts) --------

text_memory = max(
    all_stats.get("classifier_ram_after_mb", 0),
    all_stats.get("classifier_gpu_peak_mb", 0)
)

summary_rows.append({
    "pipeline": "distilroberta (gold transcripts)",

    "accuracy": all_stats.get("test_accuracy"),
    "precision": all_stats.get("test_precision"),
    "recall": all_stats.get("test_recall"),
    "f1": all_stats.get("test_f1"),

    "latency_ms": all_stats.get("classifier_latency_mean_ms"),
    "total_time_sec": all_stats.get("classifier_infer_total_sec"),

    "memory_mb": text_memory,
    "flops_per_sample": all_stats.get("approx_flops_per_sample")
})


# -------- WHISPER + ROBERTA --------

whisper_memory = max(
    pipeline_stats.get("pipeline_ram_after_mb", 0),
    pipeline_stats.get("pipeline_gpu_peak_mb", 0)
)

summary_rows.append({
    "pipeline": "whisper-small + distilroberta",

    "accuracy": pipeline_stats.get("pipeline_accuracy"),
    "precision": pipeline_stats.get("pipeline_precision"),
    "recall": pipeline_stats.get("pipeline_recall"),
    "f1": pipeline_stats.get("pipeline_f1"),

    "latency_ms": pipeline_stats.get("end_to_end_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("pipeline_total_time_sec"),

    "memory_mb": whisper_memory,
    "flops_per_sample": pipeline_stats.get("classifier_approx_flops_per_sample")
})


summary_df = pd.DataFrame(summary_rows)

print("\nFINAL SUMMARY\n")
display(summary_df)

summary_path = os.path.join(OUTPUT_DIR, "final_pipeline_comparison.csv")
summary_df.to_csv(summary_path, index=False)

print("\nSaved summary to:", summary_path)


FINAL SUMMARY



,pipeline,accuracy,precision,recall,f1,latency_ms,total_time_sec,memory_mb,flops_per_sample
0,distilroberta (gold transcripts),0.651934,0.651685,0.644444,0.648045,1.464754,0.300756,2600.101562,5586812928
1,whisper-small + distilroberta,0.602210,0.615385,0.533333,0.571429,644.007104,116.610153,2602.558594,5586812928



Saved summary to: /content/mustard_distilroberta/final_pipeline_comparison.csv


In [ ]:
summary_rows = []

summary_rows.append({
    "pipeline": "distilroberta (gold transcripts)",
    "accuracy": all_stats.get("test_accuracy"),
    "precision": all_stats.get("test_precision"),
    "recall": all_stats.get("test_recall"),
    "f1": all_stats.get("test_f1"),
    "latency_ms": all_stats.get("classifier_latency_mean_ms"),
    "total_time_sec": all_stats.get("classifier_infer_total_sec"),
    "memory_mb": max(all_stats.get("classifier_ram_after_mb", 0), all_stats.get("classifier_gpu_peak_mb", 0)),
    "flops_per_sample": all_stats.get("approx_flops_per_sample")
})

summary_rows.append({
    "pipeline": "whisper-small (ASR)",
    "accuracy": None,
    "precision": None,
    "recall": None,
    "f1": None,
    "latency_ms": pipeline_stats.get("whisper_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("whisper_total_time_sec"),
    "memory_mb": pipeline_stats.get("pipeline_gpu_peak_mb"),
    "flops_per_sample": None
})

summary_rows.append({
    "pipeline": "distilroberta (ASR transcripts)",
    "accuracy": pipeline_stats.get("pipeline_accuracy"),
    "precision": pipeline_stats.get("pipeline_precision"),
    "recall": pipeline_stats.get("pipeline_recall"),
    "f1": pipeline_stats.get("pipeline_f1"),
    "latency_ms": pipeline_stats.get("roberta_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("roberta_total_time_sec"),
    "memory_mb": pipeline_stats.get("pipeline_gpu_peak_mb"),
    "flops_per_sample": pipeline_stats.get("classifier_approx_flops_per_sample")
})

summary_rows.append({
    "pipeline": "whisper + distilroberta (e2e)",
    "accuracy": pipeline_stats.get("pipeline_accuracy"),
    "precision": pipeline_stats.get("pipeline_precision"),
    "recall": pipeline_stats.get("pipeline_recall"),
    "f1": pipeline_stats.get("pipeline_f1"),
    "latency_ms": pipeline_stats.get("end_to_end_mean_latency_ms"),
    "total_time_sec": pipeline_stats.get("pipeline_total_time_sec"),
    "memory_mb": pipeline_stats.get("pipeline_gpu_peak_mb"),
    "flops_per_sample": None
})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,pipeline,accuracy,precision,recall,f1,latency_ms,total_time_sec,memory_mb,flops_per_sample
0,distilroberta (gold transcripts),0.651934,0.651685,0.644444,0.648045,1.464754,0.300756,2600.101562,5.586813e+09
1,whisper-small (ASR),NaN,NaN,NaN,NaN,637.574106,115.400913,2029.749023,NaN
2,distilroberta (ASR transcripts),0.602210,0.615385,0.533333,0.571429,5.800882,1.049960,2029.749023,5.586813e+09
3,whisper + distilroberta (e2e),0.602210,0.615385,0.533333,0.571429,644.007104,116.610153,2029.749023,NaN


### archive

In [ ]:
import shutil

archive_path = shutil.make_archive(
    base_name="/content/mustard_distilroberta_whisper_results",
    format="zip",
    root_dir=OUTPUT_DIR
)

print("Archive created:", archive_path)

Archive created: /content/mustard_distilroberta_whisper_results.zip


### download

In [ ]:
from google.colab import files
files.download("/content/mustard_distilroberta_whisper_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>